# Day 067 · 因子的定义和分类
**Factor Taxonomy** · 阶段 P3 · 数据与因子工程

> 上一节我们搞懂了数据的两种身份,这一节正式进入因子本身。一个因子,说白了就是一把尺子,一把能给全市场所有股票横着量一遍、量完打出分数、按分数排个名的尺子。比如价值这把尺子,量的是便宜不便宜,越便宜分越高;成长这把尺子,量的是利润长得快不快,长得越猛分越高。常用的有五把尺子:价值、成长、质量、规模、动量,各量各的一面。最反直觉的地方在于,同一只股票,你换一把尺子去量,排名能天差地别,一只票可能价值尺下排第一,成长尺下却垫底。更妙的是,这五把尺子之间往往相关性很低,意思就是它们看的根本不是一回事,各管一摊。这一节我们就用八只跨风格的真实股票,亲手把这五把尺子造出来,量一遍、排个名、再看看它们彼此像不像。

---

**课件生成日期:** 2026-06-26  ·  **建议学习时长:** 20 分钟

学习路径建议:1)先看视频建立直觉 → 2)阅读本 notebook 跑代码 → 3)看 PDF 课件复习要点 → 4)做自测题

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有需要的 Python 包,缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续下面的代码

> 这一格只在第一次跑要等几十秒,后面再开 notebook 就秒过。

In [1]:
# === 环境自检 + 自动安装(运行此单元格即可) ===
# 检测缺失的库 → 自动 pip 安装 → 注入中文字体 → 一行命令搞定
import importlib
import subprocess
import sys
import os

REQUIRED = ["baostock", "matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels", "yfinance"]
PIP_NAME = {
    "sklearn": "scikit-learn",
    "cv2": "opencv-python",
    "PIL": "Pillow",
    "bs4": "beautifulsoup4",
    "yaml": "PyYAML",
}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))

if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,正在自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置(让 matplotlib 不出乱码) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

CJK_FONT_PATHS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",  # Linux/WSL
    "C:/Windows/Fonts/msyh.ttc",                               # Windows 微软雅黑
    "C:/Windows/Fonts/simhei.ttf",                             # Windows 黑体
    "/System/Library/Fonts/PingFang.ttc",                      # macOS 苹方
    "/System/Library/Fonts/STHeiti Medium.ttc",                # macOS 黑体
]
for p in CJK_FONT_PATHS:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Microsoft YaHei",
                                    "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪 — 现在可以跑下面的代码单元格")


✓ 所有 9 个必需库已就绪
✓ 中文字体已加载:NotoSansCJK-Regular.ttc
✓ 环境就绪 — 现在可以跑下面的代码单元格


## 学习目标

- 理解一个因子就是一把尺子:给全市场股票横着量一遍、打分、排名,这叫截面打分
- 认识五把最常用的尺子:价值(便宜)、成长(利润长得快)、质量(赚钱能力强)、规模(盘子小)、动量(近期强)
- 亲眼看到同一只票换把尺子排名天翻地覆:价值第一可能成长垫底,谁好谁坏全看拿哪把尺子量
- 明白因子之间相关性低意味着什么:两把尺子各看各的、不重叠,放一起用才有意义
- 建立分类直觉:见到一个新指标,先问它属于哪一类尺子、量的是股票的哪一面

## 历史背景:小X听人说要选好股票,结果把五个完全不同的标准搅成一锅粥,越选越糊涂

小X刚开始学选股,到处搜攻略。一篇说要选便宜的,市净率越低越好;一篇说要选利润涨得快的,业绩翻倍才叫好;又一篇说要选小盘股,盘子小才有爆发力;还有一篇说要追近期最强的,涨势好的才有人气。小X一股脑全记下来,挑了一只市净率很低的银行股,心想这便宜又稳,肯定是好股票。可过两天他又看到那篇讲成长的攻略,回头一查这只银行股的利润增速,慢得像蜗牛,他一下子懵了:这到底是好股票还是坏股票?同一只票,按便宜这个标准它是优等生,按长得快这个标准它又是差生,小X彻底糊涂了。他不明白的是,便宜、长得快、小盘、近期强,这是四把完全不同的尺子,量的是股票的四个不同侧面,本来就不该用一个分数概括。一只票在某把尺子下高分、另一把尺子下低分,太正常了。这节课,我们就帮小X把这几把尺子一一摆出来,告诉他每把量的是哪一面,以及为什么一只票不可能五项全优。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. 1. 什么是因子:一把给所有股票打分的尺子

因子这个词听着玄,简单讲就是一把尺子。这把尺子的用法很特别:它不是只量一只票,而是在同一个时刻,把市场上所有股票横着排开,一只一只都量一遍,量完每只票得一个分数,再按分数排个名。就好比期末考试,老师拿同一张卷子给全班同学打分,分数高的排前面。这种横着把所有票一起量、一起排名的动作,有个专门的名字叫截面打分。比如价值这把尺子,量的就是便宜不便宜:八只票里谁最便宜谁排第一。所以记住一句话:一个因子,等于说就是给全市场股票横着打分排名的一把尺子。


### 2. 2. 五把最常用的尺子各量哪一面

因子有很多,但最常用、最经典的是五把尺子,各量股票的一面。价值尺量便宜不便宜,越便宜分越高,常用市净率,也就是股价相对账面净资产的倍数。成长尺量利润长得快不快,常用净利润同比增速,翻倍的就比原地踏步的分高。质量尺量公司赚钱能力强不强,常用净资产收益率,就好比同样1 块钱本钱,谁一年能多赚出几毛。规模尺量盘子大小,小盘股分高,因为历史上小票弹性更大。动量尺量近期强不强,常用过去半年的涨幅,涨势好的分高。五把尺子,五个侧面,谁也替代不了谁。


### 3. 3. 怎么把不同尺子量出来的分数放到一起比

这里有个小麻烦:市净率可能是两倍3 倍,利润增速可能是30%,涨幅可能是10%,单位都不一样,没法直接比。解决办法叫标准化,简单讲就是把每把尺子量出来的原始数,统一换算成离平均水平多远。换算后,正分代表比平均好,负分代表比平均差,零分就是中等。就好比把语文成绩和体育成绩都换算成班级排名,这样不管原来是分数还是秒数,都能放一起比了。本节代码里,我们对每把尺子都做这个标准化,把五把尺子的分数拉到同一把刻度上,这样一只票在五把尺子下的高低,一眼就看明白。


### 4. 4. 同一只票换把尺子排名天翻地覆

这是本节最反直觉、也最重要的一点。同一只股票,你拿不同的尺子去量,排名可能完全颠倒。举个最直观的例子,一只又便宜又稳的银行股,拿价值尺量它名列前茅,因为它确实便宜;可拿成长尺量它,它利润增速很慢,立刻排到最后。反过来,一只很贵的高成长票,价值尺下它垫底,成长尺下它却是第一。所以好股票坏股票这种说法本身就不严谨,正确的问法是:在哪把尺子下好。这也解释了小X为什么糊涂,他想用一个分数概括一只票,可一只票天生就有五张面孔。


### 5. 5. 因子之间相关性低才值得一起用

既然有五把尺子,自然要问:它们彼此像不像?这就要看因子之间的相关性。相关性高,等于说两把尺子量出来的排名几乎一样,那留一把就够了,另一把是重复劳动。相关性低甚至为负,才说明两把尺子看的是完全不同的东西、各管一摊,这种组合放一起用才有价值。就好比挑队友,你不会挑五个一模一样的人,你要的是各有所长、互相补位。本节代码会把五把尺子两两的相关性都算出来,你会看到价值和成长这两把尺子往往关系很淡,因为便宜的票常常长得慢、长得快的票常常很贵,它们天然各看各的。


## 实操:造五把因子尺子(价值pbMRQ/成长YOYNI/质量roeAvg/规模mktcap/动量6月涨幅)给8只票截面z-score打分排名,再算因子两两相关性,看同一只票换尺子排名颠倒

下面这段代码跟视频里讲解的 highlights 是一致的,可以**直接 Run All** 看结果。

**依赖安装:**
```bash
pip install pandas numpy matplotlib yfinance akshare statsmodels
```


In [2]:
# day_067_factor_taxonomy.py — 因子的定义和分类:一个因子=给所有股票打分的一把尺子
# 真名上屏:baostock / query_history_k_data_plus → pbMRQ(价值) / query_profit_data → roeAvg(质量)+totalShare(算市值=规模) / query_growth_data → YOYNI(成长) / 6个月动量 / z-score 截面打分 / np.corrcoef 因子相关性
# 核心类比:一个因子就是一把尺子,把全市场股票横着量一遍打分排名;价值/成长/质量/规模/动量是五把不同的尺子;同一只票换把尺子量,排名能天翻地覆;尺子之间相关性低=各看各的
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import baostock as bs


def _data_path(_name):
    # 铁律62:data/ 放在 notebook 文件夹里。仓库根(run_lab)存取 out/notebook/data/;
    # 原版 notebook 在 out/notebook/ 用 data/;中国版在 out/notebook/cn/ 用 ../data/
    from pathlib import Path as _P
    _here = _P.cwd()
    for _b in [_here / 'data', _here / '..' / 'data', _here / 'out' / 'notebook' / 'data', _here / '..' / '..' / 'data', _here / '..' / '..' / '..' / 'data']:
        if (_b / _name).exists():
            return str(_b / _name)
    if (_here / 'out' / 'notebook').exists():
        _t = _here / 'out' / 'notebook' / 'data'
    elif _here.name == 'cn':
        _t = _here / '..' / 'data'
    else:
        _t = _here / 'data'
    _t.mkdir(parents=True, exist_ok=True)
    return str(_t / _name)


pd.set_option('display.width', 200)
plt.rcParams['axes.unicode_minus'] = False

# ==== 标的池:8 只跨风格散户熟悉的票(故意横跨价值/成长/质量/规模/动量,排名分化明显)====
STOCKS = {
    '农业银行': 'sh.601288', '亿纬锂能': 'sz.300014', '古井贡酒': 'sz.000596',
    '招商蛇口': 'sz.001979', '中国中铁': 'sh.601390', '片仔癀': 'sh.600436',
    '长安汽车': 'sz.000625', '大华股份': 'sz.002236',
}
START, END = '2022-01-01', '2024-12-31'
CSV_FACTOR = _data_path('D067_factors.csv')   # 月末因子面板:date/name/close/pbMRQ/mom6/roeAvg/YOYNI/totalShare/mktcap


# ==== 0. 拉数据:日线(close,pbMRQ)算价值+动量;季度财报(roeAvg,YOYNI,totalShare)按披露日PIT挂到月末 ====
def _fetch():
    bs.login()
    rows = []
    for name, code in STOCKS.items():
        # 日线:前复权收盘(算动量)+ pbMRQ(价值,日频自带,当天即已知)
        rs = bs.query_history_k_data_plus(code, 'date,close,pbMRQ', start_date=START, end_date=END, frequency='d', adjustflag='2')
        r = []
        while rs.error_code == '0' and rs.next():
            r.append(rs.get_row_data())
        px = pd.DataFrame(r, columns=['date', 'close', 'pbMRQ'])
        px['date'] = pd.to_datetime(px['date'])
        px['close'] = pd.to_numeric(px['close'], errors='coerce')
        px['pbMRQ'] = pd.to_numeric(px['pbMRQ'], errors='coerce')
        px = px.set_index('date').sort_index()

        # 月末快照:收盘 + 市净率 + 6个月动量(过去126个交易日涨幅)
        m_close = px['close'].resample('ME').last()
        m_pb = px['pbMRQ'].resample('ME').last()
        m_mom = px['close'].resample('ME').last() / px['close'].resample('ME').last().shift(6) - 1.0

        # 季度财报:质量 roeAvg(净资产收益率) + 总股本(算市值) ; 成长 YOYNI(净利润同比)
        # PIT:每条记录挂到 pubDate(披露日),那天才算真能拿到
        prof = []   # (pubDate, roeAvg, totalShare)
        for y in range(2021, 2025):
            for q in range(1, 5):
                rp = bs.query_profit_data(code=code, year=y, quarter=q)
                while rp.error_code == '0' and rp.next():
                    g = rp.get_row_data()   # code,pubDate,statDate,roeAvg,...,totalShare,...
                    prof.append((g[1], g[3], g[9]))
        grw = []    # (pubDate, YOYNI)
        for y in range(2021, 2025):
            for q in range(1, 5):
                rg = bs.query_growth_data(code=code, year=y, quarter=q)
                while rg.error_code == '0' and rg.next():
                    g = rg.get_row_data()   # code,pubDate,statDate,YOYEquity,YOYAsset,YOYNI,...
                    grw.append((g[1], g[5]))

        def _pit(records, col):
            d = pd.DataFrame(records, columns=['pubDate', col])
            d['pubDate'] = pd.to_datetime(d['pubDate'], errors='coerce')
            d[col] = pd.to_numeric(d[col], errors='coerce')
            d = d.dropna(subset=['pubDate']).sort_values('pubDate')
            ser = pd.Series(d[col].values, index=d['pubDate'])
            ser = ser[~ser.index.duplicated(keep='last')].sort_index()
            return ser.reindex(m_close.index, method='ffill')

        m_roe = _pit([(p, r) for p, r, _ in prof], 'roeAvg')      # prof 是 (pubDate,roeAvg,totalShare) 三元组,取前两列
        m_share = _pit([(p, s) for p, _, s in prof], 'totalShare')
        m_yoyni = _pit(grw, 'YOYNI')

        panel = pd.DataFrame({
            'name': name, 'close': m_close, 'pbMRQ': m_pb, 'mom6': m_mom,
            'roeAvg': m_roe, 'YOYNI': m_yoyni, 'totalShare': m_share,
        })
        panel['mktcap'] = panel['close'] * panel['totalShare'] / 1e8   # 总市值(亿元)
        panel.index.name = 'date'
        rows.append(panel.reset_index())
    bs.logout()
    return pd.concat(rows, ignore_index=True)


if os.path.exists(CSV_FACTOR):
    print('[数据] 从本地 CSV 读取月末因子面板')
    fac = pd.read_csv(CSV_FACTOR, parse_dates=['date'])
else:
    print('[数据] baostock 拉取 8 只票 日线(close,pbMRQ)+ 季度(roeAvg,YOYNI,totalShare),PIT 挂月末 ...')
    fac = _fetch()
    fac.to_csv(CSV_FACTOR, index=False)   # 拉完保存 CSV(铁律62)
    print('[数据] 已存 CSV ->', CSV_FACTOR)

fac['date'] = pd.to_datetime(fac['date'])

print('\n==== 因子的定义和分类:一个因子 = 给所有股票打分的一把尺子 ====')
print('标的池 %d 只 · %s ~ %s' % (len(STOCKS), fac['date'].min().date(), fac['date'].max().date()))

# ==== 1. 每个月末,把 5 个原始指标各自做成「截面 z-score 打分」(高分=这把尺子下更好/暴露更高) ====
# VALUE  价值 = z(-pbMRQ)        便宜=高分
# GROWTH 成长 = z(YOYNI)         增速高=高分
# QUALITY质量 = z(roeAvg)        赚钱能力强=高分
# SIZE   规模 = z(-log(市值))    小盘=高分
# MOM    动量 = z(过去6个月涨幅)  近期强=高分
FACTORS = ['VALUE', 'GROWTH', 'QUALITY', 'SIZE', 'MOM']
FACTOR_CN = {'VALUE': '价值', 'GROWTH': '成长', 'QUALITY': '质量', 'SIZE': '规模', 'MOM': '动量'}


def _z(s):
    s = s.astype(float)
    sd = s.std(ddof=0)
    if sd == 0 or np.isnan(sd):
        return s * 0.0
    return (s - s.mean()) / sd


scored = []   # 每个月末一张 (name × 5因子打分) 表
for d, grp in fac.groupby('date'):
    g = grp.set_index('name')
    if g['pbMRQ'].notna().sum() < len(STOCKS) or g['mktcap'].notna().sum() < len(STOCKS):
        continue
    z = pd.DataFrame(index=g.index)
    z['VALUE'] = _z(-g['pbMRQ'])
    z['GROWTH'] = _z(g['YOYNI'])
    z['QUALITY'] = _z(g['roeAvg'])
    z['SIZE'] = _z(-np.log(g['mktcap']))
    z['MOM'] = _z(g['mom6'])
    z['date'] = d
    scored.append(z)

scored = pd.concat(scored).dropna(subset=FACTORS)
month_ends = sorted(scored['date'].unique())

# ==== 2. 最新月末快照:每个因子排名 1..8,看「同一只票换把尺子排名天翻地覆」 ====
latest = month_ends[-1]
snap = scored[scored['date'] == latest][FACTORS].copy()
rank = snap.rank(ascending=False).astype(int)   # 1 = 这把尺子下最好
rank.columns = [FACTOR_CN[c] for c in FACTORS]
print('\n[最新月末 %s · 8只票 × 5因子排名表(1=该尺子下最好)]' % pd.Timestamp(latest).date())
print(rank.to_string())

# 找出「换把尺子排名落差最大」的票当头条
spread = rank.max(axis=1) - rank.min(axis=1)
star = spread.idxmax()
best_f = rank.loc[star].idxmin()
worst_f = rank.loc[star].idxmax()
print('\n[头条] %s:%s 第%d 但 %s 第%d —— 同一只票换把尺子排名天翻地覆(落差%d 名)' % (
    star, best_f, rank.loc[star].min(), worst_f, rank.loc[star].max(), spread.max()))
allrank = rank.sum(axis=1)
print('没有一只票五项全优:综合排名最靠前的 %s 五项名次合计也才 %d(满分5,垫底40)' % (allrank.idxmin(), allrank.min()))

# ==== 3. 把所有月末的打分汇成长向量,算 5×5 因子相关性(不同尺子之间像不像)====
pooled = {f: scored[f].values for f in FACTORS}
mat = np.corrcoef([pooled[f] for f in FACTORS])
corr = pd.DataFrame(mat, index=[FACTOR_CN[f] for f in FACTORS], columns=[FACTOR_CN[f] for f in FACTORS])
print('\n[5×5 因子相关性矩阵(汇总所有月末打分)]')
print(corr.round(2).to_string())

# 找一对相关性最低/最负的因子当头条
iu = np.triu_indices(len(FACTORS), k=1)
pairs = [(FACTORS[i], FACTORS[j], mat[i, j]) for i, j in zip(*iu)]
lo = min(pairs, key=lambda x: x[2])
print('[头条] 相关性最低的一对:%s 与 %s 相关系数 %.2f —— 两把尺子各看各的,几乎不重叠' % (
    FACTOR_CN[lo[0]], FACTOR_CN[lo[1]], lo[2]))

# ==== 4. 三张图 ====
# 图1:8×5 因子排名热力图(没有一只票五项全优)
fig, ax = plt.subplots(figsize=(9, 7))
R = rank.values
im = ax.imshow(R, cmap='RdYlGn_r', aspect='auto', vmin=1, vmax=len(STOCKS))
ax.set_xticks(range(len(FACTORS)))
ax.set_xticklabels([FACTOR_CN[f] for f in FACTORS])
ax.set_yticks(range(len(rank.index)))
ax.set_yticklabels(list(rank.index))
for i in range(R.shape[0]):
    for j in range(R.shape[1]):
        ax.text(j, i, str(R[i, j]), ha='center', va='center', fontweight='bold', color='black')
ax.set_title('一个因子=一把尺子:8只票×5把尺子的排名(数字=该尺子下名次,绿好红差)\n横看每只票五项颜色冷热不一——没有一只票五项全优')
fig.tight_layout()
fig.savefig('chart_01.png', dpi=110)
plt.close()

# 图2:5×5 因子相关性热力图(不同因子是不同镜头)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(mat, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(FACTORS)))
ax.set_xticklabels([FACTOR_CN[f] for f in FACTORS])
ax.set_yticks(range(len(FACTORS)))
ax.set_yticklabels([FACTOR_CN[f] for f in FACTORS])
for i in range(len(FACTORS)):
    for j in range(len(FACTORS)):
        ax.text(j, i, '%.2f' % mat[i, j], ha='center', va='center', fontweight='bold',
                color='white' if abs(mat[i, j]) > 0.55 else 'black')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('不同因子=不同镜头:5把尺子两两相关性(±1强相关,0各看各的)\n%s与%s仅%.2f——这两把尺子各管一摊,低相关才值得一起用' % (
    FACTOR_CN[lo[0]], FACTOR_CN[lo[1]], lo[2]))
fig.tight_layout()
fig.savefig('chart_02.png', dpi=110)
plt.close()

# 图3:动态挑「五因子画像最相反」的两只票(z 向量相关性最负),最能体现换尺子排名颠倒
# (不写死股票名/解说——真实数据里银行某些年成长反而高,写死会跟柱子打架)
fig, ax = plt.subplots(figsize=(11, 6))
_names = list(snap.index)
a, b, _best = _names[0], _names[1], 2.0
for _i in range(len(_names)):
    for _j in range(_i + 1, len(_names)):
        _c = np.corrcoef(snap.iloc[_i].values, snap.iloc[_j].values)[0, 1]
        if _c < _best:
            _best, a, b = _c, _names[_i], _names[_j]
za = snap.loc[a].values
zb = snap.loc[b].values
fa = FACTOR_CN[FACTORS[int(np.argmax(za))]]    # a 最拿手的尺子
fb = FACTOR_CN[FACTORS[int(np.argmax(zb))]]    # b 最拿手的尺子
x = np.arange(len(FACTORS))
w = 0.38
ax.bar(x - w / 2, za, w, color='#1565c0', label=a)
ax.bar(x + w / 2, zb, w, color='#c62828', label=b)
ax.axhline(0, c='k', lw=.8)
ax.set_xticks(x)
ax.set_xticklabels([FACTOR_CN[f] for f in FACTORS])
ax.set_ylabel('截面标准化得分 z(高=该尺子下更好)')
ax.legend()
ax.set_title('同一批票换尺子换天地:%s 在「%s」尺上领先、%s 在「%s」尺上领先,五项画像几乎相反\n谁好谁坏全看你拿哪把尺子量' % (a, fa, b, fb))
fig.tight_layout()
fig.savefig('chart_03.png', dpi=110)
plt.close()

print('\n[done] 因子=给股票打分的尺子:5把尺子排名分化 + 因子相关性 + 两只票换尺子颠倒 —— 3 图已出')

[数据] 从本地 CSV 读取月末因子面板

==== 因子的定义和分类:一个因子 = 给所有股票打分的一把尺子 ====
标的池 8 只 · 2022-01-31 ~ 2024-12-31

[最新月末 2024-12-31 · 8只票 × 5因子排名表(1=该尺子下最好)]
      价值  成长  质量  规模  动量
name                    
农业银行   2   3   4   8   1
亿纬锂能   6   6   3   4   3
古井贡酒   7   1   1   2   8
招商蛇口   3   7   8   3   2
中国中铁   1   5   6   7   6
片仔癀    8   2   2   5   5
长安汽车   5   8   7   6   7
大华股份   4   4   5   1   4

[头条] 农业银行:动量 第1 但 规模 第8 —— 同一只票换把尺子排名天翻地覆(落差7 名)
没有一只票五项全优:综合排名最靠前的 农业银行 五项名次合计也才 18(满分5,垫底40)

[5×5 因子相关性矩阵(汇总所有月末打分)]
      价值    成长    质量    规模    动量
价值  1.00 -0.13 -0.76 -0.10  0.17
成长 -0.13  1.00  0.38  0.12 -0.07
质量 -0.76  0.38  1.00  0.01 -0.10
规模 -0.10  0.12  0.01  1.00 -0.35
动量  0.17 -0.07 -0.10 -0.35  1.00
[头条] 相关性最低的一对:价值 与 质量 相关系数 -0.76 —— 两把尺子各看各的,几乎不重叠

[done] 因子=给股票打分的尺子:5把尺子排名分化 + 因子相关性 + 两只票换尺子颠倒 —— 3 图已出


## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| 换尺子排名颠倒·农业银行 | sh.601288 | 2024年12月末,农业银行在动量尺上排第1(近6个月涨得最猛),可在规模尺上排第8垫底(市值太大,规模因子偏好小票)——同一只票,两把尺子名次落差7名,谁好谁坏全看拿哪把尺子量。 |
| 好公司不便宜·古井贡酒 | sz.000596 | 古井贡酒成长第1、质量第1,赚钱又快又稳,是公认的好公司;但价值尺上排第7(市净率高、不便宜),动量尺上排第8(近期没人气)。好公司和便宜票常常是两回事,五把尺子里没有一只票全优。 |
| 因子负相关·价值vs质量 | N/A | 8只票2022-2024月度面板实测,价值因子和质量因子相关系数-0.76,是五把尺子里最负的一对:便宜(低PB)的票往往质量(ROE)偏低,贵的票往往更优质。正因为两把尺子各看各的、信息不重叠,把它们合在一起打分才有意义。 |
| 五因子镜像·古井vs招商蛇口 | N/A | 古井贡酒在质量(z+1.76)、成长(z+1.37)上高高领先,招商蛇口反而在动量(z+1.13)上领先、质量成长垫底;两只票五因子画像几乎上下镜像。同一个市场,换把尺子,谁是好票的答案完全相反。 |


## 常见坑

### ⚠ 01. 想用一个分数概括一只票,忽略它有五张面孔

一只票在价值尺下高分、成长尺下低分太正常了,好坏要看哪把尺子,别指望一个综合分说清一切。

### ⚠ 02. 把不同单位的因子直接相加比大小

市净率两倍和增速30%不在一个量纲上,不先标准化就比,等于拿米和秒比谁大,纯属乱套。

### ⚠ 03. 堆一堆高度相关的因子,以为越多越准

几把尺子相关性都很高,等于重复量同一面,看着用了五个因子其实只有一个,白费力气还自欺欺人。

### ⚠ 04. 把因子方向搞反,便宜当成贵、小盘当成大盘

价值是越便宜分越高、规模是越小盘分越高,符号一旦弄反,选出来的就是最贵最大盘的,结论整个倒过来。

### ⚠ 05. 只看某一个月的排名就给票贴死标签

因子排名是会随时间动的,这个月质量第一下个月可能滑下去,别拿一次快照当一只票永远的身份。

## 实战 SOP · 因子的定义和分类 — 几条要记牢的规矩

1. 一个因子就是一把尺子:同一时刻给全市场股票横着打分、排名,这叫截面打分。
2. 先认清五把常用尺子各量哪一面:价值(便宜)、成长(利润快)、质量(赚钱强)、规模(盘子小)、动量(近期强)。
3. 不同因子单位不同,比大小前必须先标准化,统一换算成离平均多远。
4. 别想用一个分数概括一只票:好坏要看在哪把尺子下,一只票天生有多张面孔。
5. 多因子要挑相关性低的搭配:相关性高是重复劳动,低相关才各看各的、互相补位。

> 把这段打印贴在你电脑边,执行 1000 次它会回报你。

## 总结 · 你应该带走的

2. 因子=给所有股票打分的一把尺子:拿某个指标(如市净率、利润增速、ROE)给同期所有票横向打分排名。
3. 打分方法=横截面z分:把同一个月所有票的指标减均值除标准差,不同量纲的指标也能公平比较,越高越好。
4. 五大类因子各管一个角度:价值(便不便宜·低PB)、成长(赚得快不快·利润增速)、质量(赚钱质量高不高·高ROE)、规模(大小·小市值)、动量(近期强不强·近6月涨幅)。
5. 同一只票换把尺子排名天翻地覆:农业银行动量第1却规模第8,古井贡酒成长质量双第1却价值动量垫底,没有一只票五项全优。
6. 因子之间要低相关才值得一起用:价值与质量-0.76各看各的,其余多在0附近;信息不重叠,合起来打分才不是重复下注。
7. 财报类因子(成长、质量)仍按披露日pubDate对齐(延续上一节的point-in-time),绝不用那天还没公布的财报来打分。

## 自测题

**Q1.** 用你自己的话说说:一个因子为什么可以理解成一把给所有股票打分的尺子?截面打分是什么意思?

**Q2.** 价值、成长、质量、规模、动量这五把尺子,各量的是股票的哪一面?为什么同一只票在不同尺子下排名会天翻地覆?

**Q3.** 两个因子相关性很高说明什么?为什么做多因子时我们更想要相关性低的因子组合?

把答案写下来,3 天后再回看。

## 下一节预告

**Day 068 · 价值因子** (Value Factors)

这一节我们把五把尺子都摆了出来,下一节单独深挖第一把,价值因子。我们会用市净率、市盈率、企业价值倍数三种量法,把全市场按便宜程度分组回测,看看买最便宜那一档是不是真能跑赢;同时也会揭开价值陷阱:有些票便宜得反常,是因为它真的快不行了。

## 推荐阅读

- Eugene Fama, Kenneth French《The Cross-Section of Expected Stock Returns》(1992/JoF) — 价值与规模因子的开山之作,把横截面打分这件事讲成了体系,因子投资必读。
- Andrew Ang《Asset Management: A Systematic Approach to Factor Investing》(2014/Oxford) — 系统讲因子分类与构建,五大类因子的定位说得最清楚的一本入门书。
- Cliff Asness, Andrea Frazzini, Lasse Pedersen《Quality Minus Junk》(2019/RoF) — 质量因子的代表作,讲清什么叫好公司以及怎么量化它的赚钱能力。
- Narasimhan Jegadeesh, Sheridan Titman《Returns to Buying Winners and Selling Losers》(1993/JoF) — 动量因子奠基论文,近期强者继续强这件事第一次被严格验证。
- baostock 官方文档 — 国内行情与财务接口,pbMRQ(价值)、roeAvg(质量)、YOYNI(成长)等字段对应哪把尺子,动手造因子的第一步。